In [1]:
from co_occurrence import CoOccurrenceModel
from gensim.models import Word2Vec

import gzip
import json
import numpy as np
import pandas as pd
import re
import time

# Experiment 1: Word-context representation

In [ ]:
# Khởi tạo corpus kích thước nhỏ
raw_corpus = [
    "Data mining discovers patterns in large datasets, uncovering associations and trends that may not be visible with traditional analysis techniques.",
    "Mining data helps find patterns in very large data sets, supporting decision-making processes in business, healthcare, and scientific research.",
    "Locality Sensitive Hashing finds similar documents efficiently by reducing high-dimensional vectors into compact signatures, enabling fast approximate nearest neighbor searches.",
    "MinHash approximates Jaccard similarity of sets, making it possible to compare massive collections of documents without constructing full pairwise similarity matrices.",
    "Efficient algorithms are crucial when working with massive datasets, as they allow practitioners to handle terabytes of text and multimedia data within reasonable time and memory constraints.",
    "Approximate methods trade accuracy for speed and scalability, which is essential when exact solutions are computationally infeasible in domains such as web search or large-scale recommendation.",
    "Hashing techniques are widely used in information retrieval systems, supporting fast indexing, deduplication, and similarity search across billions of documents."
]

# Hàm đếm số phần tử khác không trong word-context matrix
def count_non_zero_values(matrix: np.ndarray):
    rows, columns = matrix.shape
    count = 0
    for i in range(rows):
        for j in range(columns):
            if matrix[i, j] != 0.0:
                count += 1
    return count

# Hàm đo sparsity trên word-context matrix
def sparsity(matrix: np.ndarray):
    r, c = matrix.shape
    return round((1 - (count_non_zero_values(matrix) / (r * c))) * 100, 2)

# Train model
context_window = [1, 2, 5]

models = {
    1: CoOccurrenceModel(),
    2: CoOccurrenceModel(),
    5: CoOccurrenceModel()
}

vocab_size_list = []
matrix_shape_list = []
non_zeros_list = []
sparsity_list = []

for k in context_window:
    models[k].fit(raw_corpus, k)
    vocab_size_list.append(models[k].vocab_size)
    matrix_shape_list.append(models[k].matrix.shape)
    non_zeros_list.append(count_non_zero_values(models[k].matrix))
    sparsity_list.append(sparsity(models[k].matrix))

# Lập bảng so sánh window theo kích thước vocab, matrix, số phần tử khác 0 trong matrix và độ thưa
df = pd.DataFrame(
    [vocab_size_list, matrix_shape_list, non_zeros_list, sparsity_list],
    index = ['Vocabulary size', 'Matrix size', 'Number of non-zero values', 'Sparsity (%)'],
    columns = context_window
)
df.rename_axis('k', inplace=True)
print(df)

                                    1           2           5
k                                                            
Vocabulary size                   116         116         116
Matrix size                (116, 116)  (116, 116)  (116, 116)
Number of non-zero values         286         562        1256
Sparsity (%)                    97.87       95.82       90.67


- Dễ thấy, khi tăng context window, số phần tử khác 0 của word-context matrix ngày càng tăng, trong khi độ thưa (sparsity) có xu hướng giảm, khi k = 1 thì sparsity bằng 97.87% nhưng khi k = 5 thì sparsity chỉ còn 90.67%.

In [ ]:
# Lấy 5 từ có similarity cao nhất với từ 'data'
for k in context_window:
    print(models[k].most_similar('data', 5))

[('discovers', np.float64(0.4714045207910316)), ('find', np.float64(0.2357022603955158)), ('making', np.float64(0.2357022603955158)), ('reasonable', np.float64(0.2357022603955158)), ('very', np.float64(0.2357022603955158))]
[('patterns', np.float64(0.5715476066494082)), ('helps', np.float64(0.3872983346207417)), ('time', np.float64(0.3872983346207417)), ('in', np.float64(0.26352313834736496)), ('decision-making', np.float64(0.2581988897471611))]
[('helps', np.float64(0.7381017469576844)), ('large', np.float64(0.7149000896634314)), ('mining', np.float64(0.6315089779506726)), ('find', np.float64(0.5763708787148328)), ('very', np.float64(0.5637345210021217))]


- Khi tăng giá trị context window, similarity giữa context word vector và target word vector ngày càng tăng. Điều này cho thấy khi mô hình học được lượng ngữ cảnh nhiều hơn thì lượng thông tin tiếp nhận sẽ tăng và có thể tìm ra các từ có thể similarity cao hơn.

## Từ co-occurence matrix đến embedding

- Nếu vocabulary có 100,000 từ thì kích thước word-context matrix sẽ rất lớn và thưa, cỡ 100,000 x 100,000, gây ra sự tiêu tốn trong lưu trữ bộ nhớ và việc tính toán sẽ chậm hơn nhiều, nhất là semantic similarity.

## Khác biệt giữa Skip-gram và CBOW

- Với CBOW, input là các từ xét trong cùng 1 context cụ thể, trong khi target của nó là 1 từ mục tiêu dự đoán trên context đang xét.
- Với Skip-gram thì ngược lại, input là một target word, trong khi target của nó là dự đoán các context words dựa theo thông tin của target word đã cho.

# Experiment 2: Word2Vec

In [22]:
data_path = '../data/c4-train.00000-of-01024-30K.json.gz'

# Đọc dữ liệu
raw_corpus = []

with gzip.open(data_path, 'rt', encoding = 'utf-8') as f:
    for line in f:
        data = json.loads(line)
        raw_corpus.append(data['text'])

print(f'Kích thước corpus: {len(raw_corpus)}')

# Tiền xử lý dữ liệu
## Hàm tách câu
def split_sentences(document):
    return [
        sentence.strip()
        for sentence in re.split(r'(?<=[.!?])\s+|\n+', document)
        if sentence.strip()
    ]

## Hàm tokenize
def tokenize(sentence):
    sentence = sentence.lower()
    return re.findall(
        r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
        sentence
    )

# Tập document đã được tokenize
tokenized_documents = []
# Tập các câu đã được tokenize
tokenized_sentences = []
# Tập từ vựng
vocabulary = set()

# Bước tokenize
for document in raw_corpus:
    document_tokens = []

    for sentence in split_sentences(document):
        tokens = tokenize(sentence)

        if tokens:
            tokenized_sentences.append(tokens)
            vocabulary.update(tokens)
            document_tokens.extend(tokens)

    if document_tokens:
        tokenized_documents.append(document_tokens)

vocabulary = sorted(vocabulary)

print(f'Số câu: {len(tokenized_sentences)}')
print(f'Kích thước vocabulary: {len(vocabulary)}')

Kích thước corpus: 30000
Số câu: 634941
Kích thước vocabulary: 227644


- Mô tả cấu hình huấn luyện model Word2Vec:
    + `sentences = tokenize_sentences`: Tập các câu đã được tokenize từ corpus để đưa vào huấn luyện. Ví dụ: `[['I', 'likes', 'NLP'], ['I', 'study', 'maths'], ...]`
    + `vector_size = 100`: Số chiều của embedding vectors. Trong phần này, ta sẽ nhúng mỗi từ bằng vector dense 100 chiều.
    + `window = 5`: Kích thước window cho model. Ở đây, window = 5 để xét 5 token bên trái và 5 token bên phải cho center word làm context.
    + `min_count = 2`: Giữ lại những token xuất hiện ít nhất 2 lần để huấn luyện model.
    + `epochs = 10`: Toàn bộ dữ liệu đưa vào model 10 lần để huấn luyện.
    + `sg = 0`: sg = 0: CBOW, sg = 1: Skip-gram, training objective: CBOW
    + `seed = 42`: Cố định kết quả ở mỗi lần chạy code.

In [20]:
model_w2v = Word2Vec(
    sentences = tokenized_sentences,
    vector_size = 100,
    window = 5,
    min_count = 2,
    epochs = 10,
    sg = 0,
    seed = 42
)

- Inspect embeddings

In [21]:
inspect_word = ['doctor', 'hospital', 'patient', 'disease', 'computer', 'football', 'banana']
top_5_list = []
for word in inspect_word:
    top_5 = model_w2v.wv.most_similar(word, topn = 5)
    top_5_word = [w[0] for w in top_5]
    top_5_list.append(top_5_word)

df_top_5 = pd.DataFrame(
    top_5_list,
    index = inspect_word,
    columns = [f'top_{k}' for k in range(1, 6)]
)

df_top_5

,top_1,top_2,top_3,top_4,top_5
doctor,dentist,veterinarian,physician,therapist,surgeon
hospital,clinic,clinics,hospitals,daycare,hospice
patient,patients,physician,client,diagnosis,child
disease,diseases,infection,chronic,obesity,diabetes
computer,desktop,device,laptop,computers,server
football,baseball,soccer,basketball,replica,nhl
banana,strawberries,molasses,cheddar,caramelized,chutney


- Ví dụ với `doctor`: Dễ thấy các từ như `dentist`, `physician`, ... có semantic similarity cao nhất với `doctor` vì những từ này xuất hiện nhiều trong cùng các context tương tự nhau, nhất là context liên quan đến y khoa. Chẳng hạn, `doctor` và các từ trong top-5 thường đi kèm với các context word như `patient`, `hospital`, `medical`,... hay các context word có liên quan đến y khoa. Model Word2Vec học representation từ các context theo window, dẫn đến có sự lặp lại và overlap giữa các context khi xuất hiện trong các training example khi huấn luyện model. Do đó, các embedding vectors của các từ nói trên có cấu trúc gần nhau hơn, nên semantic similarity của chúng sẽ cao

# Experiment 3: Context window

- Train trên 3 model với context window lần lượt là 2, 5, 10 và lưu bảng so sánh similarity giữa cặp từ theo window.

In [7]:
window_size = [2, 5, 10]
word_pair = [
    ('doctor', 'physician'),
    ('doctor', 'hospital'),
    ('cat', 'dog')
]
window_results = []

for k in window_size:
    model = Word2Vec(
        sentences = tokenized_sentences,
        vector_size = 100,
        min_count = 2,
        window = k,
        epochs = 10,
        seed = 42
    )
    window_similarity = []

    for first_word, second_word in word_pair:
        window_similarity.append(model.wv.similarity(first_word, second_word))

    window_results.append(window_similarity)

df_window = pd.DataFrame(
    list(zip(*window_results)),
    columns = [f'Window {k}' for k in window_size],
    index = ['doctor-physician', 'doctor-hospital', 'cat-dog']
)

df_window

,Window 2,Window 5,Window 10
doctor-physician,0.790248,0.782484,0.778612
doctor-hospital,0.481452,0.480788,0.476436
cat-dog,0.708909,0.790064,0.772463


- Qua việc khảo sát context window trên các cặp từ, similarity của một số cặp từ có phần giảm khi tăng context window, điều này cho thấy window lớn hơn chưa chắc giúp mô hình tốt hơn, vì nó có thể khiến mô hình học thêm lượng thông tin nhiễu.

# Experiment 4: Embedding dimension

- So sánh và đánh giá mô hình theo embedding dimensions với độ dài là 50, 100, 300.

In [10]:
dimensions = [50, 100, 300]

training_time = []
model_size = []
dim_similarity = []
analogy = []
models_by_dimension = {}

for dim_value in dimensions:
    start = time.perf_counter()

    model = Word2Vec(
        sentences = tokenized_sentences,
        vector_size = dim_value,
        window = 5,
        min_count = 2,
        sg = 1,
        workers = 1,
        seed = 42,
        epochs = 10
    )

    elapsed = time.perf_counter() - start
    training_time.append(elapsed)
    models_by_dimension[dim_value] = model

    model_size.append(model.wv.vectors.nbytes)

    dim_similarity_pair = []
    for first_word, second_word in word_pair:
        dim_similarity_pair.append(model.wv.similarity(
            first_word, second_word
        ))
    dim_similarity.append(dim_similarity_pair)

    analogy.append(model.wv.most_similar(
        positive = ['king', 'woman'],
        negative = ['man'],
        topn = 5
    ))

df_eval_dim = pd.DataFrame(
    [training_time, model_size],
    index = ['Training time', 'Model size'],
    columns = dimensions
)
df_eval_dim.rename_axis('Embedding dimension', inplace=True)

df_eval_dim

,50,100,300
Embedding dimension,,,
Training time,2.346096e+03,1.770141e+03,2.084509e+03
Model size,2.178500e+07,4.357000e+07,1.307100e+08


- Model size của model tăng đều khi dimension tăng, trong khi thời gian huấn luyện model có xu hướng dao động khi tăng dimension, trong đó training time của model Word2Vec với 100 chiều mất 1770s, trong khi model Word2Vec với 300 chiều mất đến 2084s.

- So sánh theo similarity giữa các word pair.

In [11]:
df_similarity_dim = pd.DataFrame(
    list(zip(*dim_similarity)),
    columns = dimensions,
    index = ['doctor-physician', 'doctor-hospital', 'cat-dog']
)
df_similarity_dim.rename_axis('Embedding dimension', inplace=True)
df_similarity_dim

,50,100,300
Embedding dimension,,,
doctor-physician,0.825215,0.731052,0.573056
doctor-hospital,0.623291,0.563873,0.387212
cat-dog,0.860076,0.780533,0.536428


- Khi huấn luyện model với dimension ngày càng tăng, similarity của các từ có xu hướng ngày càng giảm, có thể do model học các training example nhiễu, hoặc có các context không phù hợp và khác biệt.

- Đánh giá dựa trên analogy

In [12]:
analogy_rows = []

for dimension, results in zip(dimensions, analogy):
    for rank, (word, similarity) in enumerate(results, start=1):
        analogy_rows.append({
            'Dimension': dimension,
            'Rank': rank,
            'Top-5 word': word,
            'Similarity': similarity
        })

df_analogy = pd.DataFrame(analogy_rows)
df_analogy

,Dimension,Rank,Top-5 word,Similarity
0,50,1,queen,0.888257
1,50,2,victoria,0.714008
2,50,3,prince,0.712427
3,50,4,albert,0.706699
4,50,5,royal,0.705625
5,100,1,queen,0.780345
6,100,2,queen's,0.560841
7,100,3,duke,0.549678
8,100,4,duchess,0.537112
9,100,5,lila,0.532188


- Từ `Queen` đều nằm ở rank 1 trong Top-5 trên cả 3 dimension. Điều này cho thấy phép tính `king - man + woman` tạo ra một vector gần với `queen` trong embedding. Tuy nhiên, đây chỉ là một pattern được học từ context của corpus.

- Downstream task: Phân loại các từ theo chủ đề medical và sport

In [13]:
medical_words = ['doctor', 'hospital', 'patient', 'disease']
sports_words = ['football', 'baseball', 'basketball', 'hockey']
test_words = {
    'physician': 'medical',
    'clinic': 'medical',
    'infection': 'medical',
    'soccer': 'sports',
    'nba': 'sports'
}

def cosine_similarity(vector1, vector2):
    denominator = np.linalg.norm(vector1) * np.linalg.norm(vector2)
    if denominator == 0:
        return 0.0
    return float(np.dot(vector1, vector2) / denominator)

downstream_rows = []
for dimension, model in models_by_dimension.items():
    medical = [word for word in medical_words if word in model.wv]
    sports = [word for word in sports_words if word in model.wv]
    medical_centroid = np.mean([model.wv[word] for word in medical], axis=0)
    sports_centroid = np.mean([model.wv[word] for word in sports], axis=0)

    for word, expected in test_words.items():
        if word not in model.wv:
            downstream_rows.append({
                'dimension': dimension, 'word': word,
                'expected': expected, 'predicted': 'OOV', 'correct': False
            })
            continue

        medical_score = cosine_similarity(model.wv[word], medical_centroid)
        sports_score = cosine_similarity(model.wv[word], sports_centroid)
        predicted = 'medical' if medical_score >= sports_score else 'sports'
        downstream_rows.append({
            'dimension': dimension, 'word': word,
            'expected': expected, 'predicted': predicted,
            'medical_score': medical_score, 'sports_score': sports_score,
            'correct': predicted == expected
        })

df_downstream_detail = pd.DataFrame(downstream_rows)
df_downstream_detail

,dimension,word,expected,predicted,medical_score,sports_score,correct
0,50,physician,medical,medical,0.827107,0.309382,True
1,50,clinic,medical,medical,0.810226,0.384891,True
2,50,infection,medical,medical,0.770317,0.195619,True
3,50,soccer,sports,sports,0.276256,0.886889,True
4,50,nba,sports,sports,0.238656,0.798841,True
5,100,physician,medical,medical,0.741741,0.258897,True
6,100,clinic,medical,medical,0.759265,0.318941,True
7,100,infection,medical,medical,0.659733,0.168261,True
8,100,soccer,sports,sports,0.255145,0.808296,True
9,100,nba,sports,sports,0.257922,0.723724,True


- Đánh giá độ chính xác trên downstream task

In [14]:
df_downstream_summary = (
    df_downstream_detail.assign(
        evaluated=lambda data: data['predicted'] != 'OOV'
    )
    .groupby('dimension')
    .agg(evaluated=('evaluated', 'sum'), correct=('correct', 'sum'))
    .assign(accuracy=lambda data: data['correct'] / data['evaluated'])
)
df_downstream_summary

,evaluated,correct,accuracy
dimension,,,
50,5,5,1.0
100,5,5,1.0
300,5,5,1.0


- Model phân loại đúng toàn bộ các từ trong tập kiểm tra nhỏ. Tuy nhiên, kết quả này chỉ có ý nghĩa minh họa vì số lượng test word ít và các centroid được tạo từ chính các từ đại diện cho hai nhóm.

# Evaluation

## Evaluation — Word similarity

Đánh giá các cặp từ bằng cosine similarity trên ba embedding dimensions.

In [15]:
evaluation_pairs = [
    ('doctor', 'physician'),
    ('car', 'automobile'),
    ('king', 'queen'),
    ('cat', 'dog'),
    ('computer', 'banana')
]

similarity_rows = []
for first_word, second_word in evaluation_pairs:
    row = {'word_pair': f'{first_word}-{second_word}'}
    for dimension, model in models_by_dimension.items():
        if first_word in model.wv and second_word in model.wv:
            row[dimension] = model.wv.similarity(first_word, second_word)
        else:
            row[dimension] = np.nan
    similarity_rows.append(row)

df_word_similarity = pd.DataFrame(similarity_rows).set_index('word_pair')
df_word_similarity.rename_axis('Embedding dimension', axis=1, inplace=True)
df_word_similarity

Embedding dimension,50,100,300
word_pair,,,
doctor-physician,0.825215,0.731052,0.573056
car-automobile,0.824350,0.734005,0.532337
king-queen,0.894611,0.779207,0.482665
cat-dog,0.860076,0.780533,0.536428
computer-banana,0.021134,0.005028,0.078055


Giá trị `NaN` biểu thị từ OOV (không có trong vocabulary của model), không phải similarity bằng 0.

- Kết quả phù hợp với intuition: các từ cùng lĩnh vực hoặc xuất hiện trong context tương tự có cosine similarity cao hơn, chẳng hạn như `king-queen`, `cat-dog`
- Similarity giữa các từ giảm dần khi tăng context window, do model học các training examples nhiễu.
- `doctor` và `hospital` không phải từ đồng nghĩa, nhưng cùng xuất hiện trong context lien quan đến y khoa, do Word2Vec tập trung nhiều vào semantic similarity.
- Cặp `computer-banana` có semantic similarity thấp nhất, nhưng lại nhỉnh hơn so với intuition, có thể do 2 từ xuất hiện trong các context đặc thù của corpus, chẳng hạn document về sản phẩm hoặc thực phẩm, hoặc do vector trung bình bị ảnh hưởng bởi các context phổ biến. Điều này cho thấy similarity không phải lúc nào cũng phản ánh quan hệ ngữ nghĩa theo phán đoán của con người.

## Evaluation — Word analogy

Kiểm tra `king - man + woman ≈ queen` bằng Top-5 kết quả.

In [16]:
analogy_input = ['king', 'man', 'woman']
analogy_rows = []

for dimension, model in models_by_dimension.items():
    missing_words = [word for word in analogy_input if word not in model.wv]
    if missing_words:
        analogy_rows.append({
            'dimension': dimension,
            'rank': np.nan,
            'word': 'OOV: ' + ', '.join(missing_words),
            'similarity': np.nan,
            'queen_rank': np.nan
        })
        continue

    results = model.wv.most_similar(
        positive=['king', 'woman'],
        negative=['man'],
        topn=5
    )
    queen_rank = next(
        (rank for rank, (word, _) in enumerate(results, start=1)
         if word == 'queen'),
        np.nan
    )

    for rank, (word, score) in enumerate(results, start=1):
        analogy_rows.append({
            'dimension': dimension,
            'rank': rank,
            'word': word,
            'similarity': score,
            'queen_rank': queen_rank
        })

df_word_analogy = pd.DataFrame(analogy_rows)
df_word_analogy

,dimension,rank,word,similarity,queen_rank
0,50,1,queen,0.888257,1
1,50,2,victoria,0.714008,1
2,50,3,prince,0.712427,1
3,50,4,albert,0.706699,1
4,50,5,royal,0.705625,1
5,100,1,queen,0.780345,1
6,100,2,queen's,0.560841,1
7,100,3,duke,0.549678,1
8,100,4,duchess,0.537112,1
9,100,5,lila,0.532188,1


- Việc `queen` luôn đứng đầu khi đánh giá analogy của `king - man + woman` thể hiện vector arithmetic có thể biểu diễn một số quan hệ vì embedding space học từ nhiều context tương tự trong corpus. Những cặp từ có quan hệ tương tự thường sẽ có nhận dạng dịch chuyển tương tự về hướng hoặc gần như song song với nhau, chẳng hạn: quan hệ `man -> king` tương tự như `woman -> queen`.

- Lưu kết quả trong `results.csv`

In [ ]:
result_tables = []

def add_result_table(dataframe, section):
    table = dataframe.reset_index()
    table.insert(0, 'section', section)
    result_tables.append(table)

add_result_table(df, 'experiment_1')
add_result_table(df_top_5, 'experiment_2_top_5_similar_word')
add_result_table(df_window, 'experiment_3_context_window')
add_result_table(df_eval_dim, 'experiment_4_cost')
add_result_table(df_similarity_dim, 'experiment_4_similarity')
add_result_table(df_analogy, 'experiment_4_analogy')
add_result_table(df_downstream_detail, 'experiment_4_downstream_detail')
add_result_table(df_downstream_summary, 'experiment_4_downstream_summary')
add_result_table(df_word_similarity, 'evaluation_word_similarity')
add_result_table(df_word_analogy, 'evaluation_word_analogy')

results = pd.concat(result_tables, ignore_index=True, sort=False)
results.to_csv('../results.csv', index=False, encoding='utf-8-sig')
print(f'Đã lưu {len(results)} dòng vào ../results.csv')
results.head()

# Application - Semantic Search

- API cho tìm kiếm ngữ nghĩa

In [61]:
def get_known_tokens(tokens: list[str], model: Word2Vec):
    known_tokens = []
    oov_tokens = []

    for token in tokens:
        if token in model.wv:
            known_tokens.append(token)
        else:
            oov_tokens.append(token)

    return known_tokens, oov_tokens

def mean_embedding(tokens: list[str], model: Word2Vec):
    known_tokens, oov_tokens = get_known_tokens(tokens, model)

    if not known_tokens:
        return None, oov_tokens

    vectors = np.array([
        model.wv[token]
        for token in known_tokens
    ])

    mean_vector = np.mean(vectors, axis = 0)

    return mean_vector, oov_tokens

def cosine_similarity(vector1, vector2):
    if vector1 is None or vector2 is None:
        return 0.0
    
    denominator = (
        np.linalg.norm(vector1) * np.linalg.norm(vector2)
    )
    if denominator == 0.0:
        return 0.0
    return float(np.dot(vector1, vector2) / denominator)

def build_document_vectors(
    tokenized_documents: list[list[str]], 
    model: Word2Vec,
    raw_documents: list[str] | None = None
):
    
    document_vectors = []

    for document_index, tokens in enumerate(tokenized_documents):
        mean_vector, oov_tokens = mean_embedding(
            tokens,
            model
        )

        document_result = {
            'document_index': document_index,
            'vector': mean_vector,
            'oov_tokens': oov_tokens
        }

        if raw_documents is not None:
            document_result["text"] = raw_documents[document_index]

        document_vectors.append(document_result)

    return document_vectors

def tokenize(text: str):
    return re.findall(
        r"[A-Za-z0-9]+(?:[-'][A-Za-z0-9]+)*",
        text.lower()
    )

def semantic_search(
    query: str,
    tokenized_documents: list[list[str]],
    model: Word2Vec, 
    raw_documents: list[str] | None = None,
    top_k: int = 5
):
    query_tokens = tokenize(query)
    query_vector, query_oov_tokens = mean_embedding(
        query_tokens,
        model
    )

    document_vectors = build_document_vectors(
        tokenized_documents,
        model,
        raw_documents
    )

    results = []

    for document in document_vectors:
        if document["vector"] is None:
            continue

        score = cosine_similarity(
            query_vector,
            document['vector']
        )

        result = {
            'document_index': document['document_index'],
            'score': score,
            'oov_tokens': document['oov_tokens']
        }

        if raw_documents is not None:
            result['text'] = document['text']

        results.append(result)

    results.sort(
        key = lambda result: result['score'],
        reverse = True
    )

    results = results[:top_k]

    for rank, result in enumerate(results, start=1):
        result["rank"] = rank

    return {
        'query': query,
        'query_oov_tokens': query_oov_tokens,
        'results': results
    }

- Tiến hành với query `medical treatment`

In [62]:
search_output = semantic_search(
    query = 'medical treatment', 
    tokenized_documents = tokenized_documents, 
    model = models_by_dimension[50], 
    raw_documents = raw_corpus
)
df_semantic_search = pd.DataFrame(search_output['results'])

df_semantic_search

,document_index,score,oov_tokens,text,rank
0,280,0.870327,[opioid-refractory],A phase II pilot study to evaluate use of intr...,1
1,9420,0.858957,[],Treatment of bleeding problems without removal...,2
2,22684,0.855159,"[aorto-bifemoral, femoro-femoral, femoro-popli...",The Indiana Hospital & Heart Institute has one...,3
3,8437,0.849168,"[vaporhcs, patient-clinician]",Dr. Slatore is a Pulmonary/Critical Care physi...,4
4,3666,0.833827,"[pre-clinic, orofacial]",Specific focus on the use of emerging technolo...,5


# Error Analysis

In [63]:
search_output_1 = semantic_search(
    query = 'soccer game', 
    tokenized_documents = tokenized_documents, 
    model = models_by_dimension[50], 
    raw_documents = raw_corpus,
    top_k = 10
)
df_semantic_search_1 = pd.DataFrame(search_output_1['results'])

print(df_semantic_search_1)

   document_index     score                   oov_tokens  \
0           24068  0.824244                   [tennis24]   
1           23546  0.781610                           []   
2           25572  0.779794                           []   
3            1915  0.768637                  [gamecocks]   
4           16031  0.766893                     [pikpok]   
5           29155  0.764484                     [cabela]   
6            1326  0.764464                           []   
7           20399  0.762038  [livescores, tournaments's]   
8            5478  0.757877                           []   
9           29560  0.757203                           []   

                                                text  rank  
0  Help: You are on Tennis Reunion Island (Challe...     1  
1  lets play again league of legends part 7: Prob...     2  
2  Baseball Factory, the Official Player Developm...     3  
3  Sports Traveler has your South Carolina Gameco...     4  
4  "Become a goalkeeper and lead y

In [64]:
index_3 = [d.get('document_index') for d in search_output_1['results']]
for i in index_3:
    print(raw_corpus[i])
    print()

Help: You are on Tennis Reunion Island (Challenger Men - Singles) page. Live tennis results at Tennis 24 offer tennis live scores and results, providing also set results, H2H stats and other tennis live score information from Challenger Men - Singles - Reunion Island and more than 400 other tennis tournaments. Get tennis Reunion Island scores, live ATP tennis, live WTA tournaments, Challengers online, ITF tournaments - all at Tennis24.com.

lets play again league of legends part 7: Problems. . .

Baseball Factory, the Official Player Development Partner of Little League Baseball®, is thrilled to be back at the Little League Baseball® World Series to create videos and scouting reports for each participant.
Check out the latest Instagrams from the Little League World Series.
©2018 Little League Baseball, Incorporated. All Rights Reserved.

Sports Traveler has your South Carolina Gamecocks Tickets to the game! Purchase securely online in our real time inventory. Follow the team to the Col

# Polysemy

- Static embedding chỉ gán một vector cố định cho mỗi từ. Với từ đa nghĩa như `bank`, vector sẽ được tổng hợp từ việc học tất cả các context xuất hiện, khiến nó trở thành representation của nhiều nghĩa. Đó là hạn chế của static embeddings trong việc phân biệt polysemy, và do đó những mô hình contextual embeddings như BERT ra đời để giải quyết vấn đề này.

# Individual learning check

- Tại sao tăng context window có thể vừa tốt vừa xấu?
    + Context window có thể lấy các từ liền cạnh bên trái hoặc bên phải của centre word. Khi tăng context window, model có thể học thêm nhiều context quan trọng, giúp tăng thông tin về ngữ cảnh của từ đang xét. Tuy vậy, nhược điểm khi tăng context window là model có thể học các training examples nhiễu, không thích hợp hoặc liên quan với context của từ.